# ByteBank — Detecção de fraudes em cartões de crédito
**Desafio de Projeto | Python, pandas, NumPy, scikit-learn, XGBoost e SHAP**

Objetivo: comparar três modelos, priorizar **recall da fraude**, medir o custo dos falsos positivos, selecionar um limiar na **validação** e avaliar **uma única vez** no teste.

> Projeto educacional. O dataset é histórico, anonimizado e não representa um sistema bancário pronto para produção. Resultados não foram pré-calculados.

## 1. Aquisição e exploração dos dados
O dataset original tem 284.807 linhas e 492 fraudes. O download é feito diretamente do Hugging Face (espelho público). O arquivo é grande (~151 MB).

In [ ]:
from pathlib import Path
import hashlib
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
DATA.mkdir(exist_ok=True, parents=True)
RESULTS.mkdir(exist_ok=True, parents=True)
FILE = DATA / "creditcard.csv"
URL = "https://huggingface.co/datasets/David-Egea/Creditcard-fraud-detection/resolve/main/creditcard.csv"
EXPECTED_SHA256 = "76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89"

if not FILE.exists():
    print("Baixando dataset público (~151 MB)...")
    urllib.request.urlretrieve(URL, FILE)

sha = hashlib.file_digest(FILE.open("rb"), "sha256").hexdigest()
if sha != EXPECTED_SHA256:
    raise ValueError("SHA-256 inesperado: confirme a origem ou substitua o arquivo.")
df = pd.read_csv(FILE)
required = {"Time", "Amount", "Class"} | {f"V{i}" for i in range(1, 29)}
assert required.issubset(df.columns), "Colunas esperadas ausentes."
assert set(df["Class"].unique()) == {0, 1}
print("Dimensões:", df.shape)
print("Fraudes:", int(df.Class.sum()), "| Taxa:", f"{df.Class.mean():.4%}")
print("Valores ausentes:", int(df.isna().sum().sum()))
display(df.head())

In [ ]:
counts = df["Class"].value_counts().sort_index()
ax = counts.rename(index={0: "Legítima", 1: "Fraude"}).plot.bar(title="Distribuição das classes")
ax.set_ylabel("Número de transações")
plt.tight_layout()
plt.show()
display(df[["Time", "Amount"]].describe())
print("Acurácia de prever sempre legítima:", f"{(df.Class == 0).mean():.4%}")

## 2. Preparação sem vazamento
Divisão estratificada **60% treino / 20% validação / 20% teste**. A escala é aprendida somente no treino via `Pipeline`. Usamos `class_weight='balanced'` ou `scale_pos_weight` para lidar com o desbalanceamento. Não fazemos SMOTE antes da divisão.

**Limitação:** esta é uma avaliação aleatória estratificada, não uma validação temporal nem uma avaliação por cliente. Um teste de produção exigiria validação cronológica, identificação de entidades e monitorização de deriva.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_curve,
    PrecisionRecallDisplay, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

X = df.drop(columns="Class")
y = df["Class"].astype(int)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print("Treino/Validação/Teste:", len(X_train), len(X_val), len(X_test))
print("Fraudes por divisão:", y_train.sum(), y_val.sum(), y_test.sum())

## 3. Treinamento e comparação
A seleção preliminar usa **Average Precision (AP, resumo da curva precision-recall)** na validação. Recall sozinho, sem restrições, favoreceria marcar todas as operações como fraude.

In [ ]:
imbalance = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    "Regressão Logística": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=1500, random_state=SEED
        ))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=SEED
    ),
    "XGBoost": XGBClassifier(
        n_estimators=250, max_depth=4, learning_rate=0.07,
        subsample=0.85, colsample_bytree=0.85,
        scale_pos_weight=float(imbalance),
        eval_metric="logloss", tree_method="hist", n_jobs=4,
        random_state=SEED
    )
}
scores_val = {}
rows = []
for name, model in models.items():
    print("Treinando:", name)
    model.fit(X_train, y_train)
    p = model.predict_proba(X_val)[:, 1]
    scores_val[name] = p
    rows.append({
        "modelo": name,
        "AP_validacao": average_precision_score(y_val, p),
        "recall_0_5": recall_score(y_val, p >= 0.5, zero_division=0),
        "precision_0_5": precision_score(y_val, p >= 0.5, zero_division=0),
    })
comparison = pd.DataFrame(rows).sort_values("AP_validacao", ascending=False)
display(comparison)
comparison.to_csv(RESULTS / "comparacao_validacao.csv", index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for name, probs in scores_val.items():
    PrecisionRecallDisplay.from_predictions(y_val, probs, name=name, ax=ax)
ax.set_title("Precision–Recall na validação")
plt.tight_layout()
plt.savefig(RESULTS / "precision_recall_validacao.png", dpi=160)
plt.show()

## 4. Escolha do limiar e avaliação final
Critério operacional **explícito e ajustável**: maximizar recall sujeito a `precision >= 0.20` na validação. Se nenhum limiar satisfizer esse requisito, o notebook sinaliza a impossibilidade e escolhe o maior F1 na validação como *fallback educacional*, sem fingir que a restrição foi cumprida.

A seleção conjunta de modelo e limiar acontece **somente na validação**. O teste fica intocado até o final.

In [ ]:
MIN_PRECISION = 0.20  # hipótese didática; substituir por restrição de negócio
def choose_threshold(y_true, probs, min_precision=MIN_PRECISION):
    precision, recall, thresholds = precision_recall_curve(y_true, probs)
    # O último par precision/recall não tem threshold correspondente.
    p, r = precision[:-1], recall[:-1]
    candidates = np.flatnonzero(p >= min_precision)
    if len(candidates):
        # Maior recall; desempate por precision e limiar maior.
        best = max(candidates, key=lambda i: (r[i], p[i], thresholds[i]))
        feasible = True
    else:
        f1 = 2 * p * r / np.maximum(p + r, 1e-12)
        best = int(np.argmax(f1))
        feasible = False
    return {
        "threshold": float(thresholds[best]),
        "precision": float(p[best]),
        "recall": float(r[best]),
        "feasible": feasible,
    }

candidates = []
for name, probs in scores_val.items():
    picked = choose_threshold(y_val, probs)
    candidates.append({
        "modelo": name, "AP_validacao": average_precision_score(y_val, probs),
        **picked
    })
thresholds_table = pd.DataFrame(candidates)
display(thresholds_table.sort_values(["feasible", "recall", "precision", "AP_validacao"],
                                      ascending=False))
eligible = thresholds_table[thresholds_table.feasible]
pool = eligible if not eligible.empty else thresholds_table
winner = pool.sort_values(
    ["recall", "precision", "AP_validacao"], ascending=False
).iloc[0]
best_name = str(winner["modelo"])
best_threshold = float(winner["threshold"])
print("Selecionado:", best_name, "| limiar:", best_threshold)
if eligible.empty:
    print("AVISO: nenhum modelo cumpriu a precisão mínima definida.")
thresholds_table.to_csv(RESULTS / "limiares_validacao.csv", index=False)

In [ ]:
# Única avaliação final no teste (sem recalibrar o limiar)
best_model = models[best_name]
test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)
metrics = {
    "modelo": best_name,
    "threshold": best_threshold,
    "AP_teste": float(average_precision_score(y_test, test_prob)),
    "recall_fraude": float(recall_score(y_test, test_pred, zero_division=0)),
    "precision_fraude": float(precision_score(y_test, test_pred, zero_division=0)),
    "f1_fraude": float(f1_score(y_test, test_pred, zero_division=0)),
}
display(pd.DataFrame([metrics]))
print(classification_report(y_test, test_pred, zero_division=0))
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
print("TN, FP, FN, TP:", cm[0,0], cm[0,1], cm[1,0], cm[1,1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legítima", "Fraude"]).plot()
plt.title("Matriz de confusão — teste")
plt.tight_layout()
plt.savefig(RESULTS / "matriz_confusao_teste.png", dpi=160)
plt.show()
pd.DataFrame([metrics]).to_csv(RESULTS / "metricas_teste.csv", index=False)

## 5. Explicabilidade com SHAP
Explicamos o modelo vencedor em **amostra pequena do teste** para evitar custo excessivo. O SHAP descreve a influência das variáveis nas previsões, **não causalidade**. Como `V1`–`V28` são componentes PCA anonimizados, sua interpretação de negócio é limitada.

In [ ]:
import shap

sample = X_test.sample(n=min(100, len(X_test)), random_state=SEED)
background = X_train.sample(n=min(100, len(X_train)), random_state=SEED)
# Abordagem genérica e consistente: explicar a probabilidade de fraude.
# Pode ser mais lenta que TreeExplainer, mas funciona para os três modelos.
predict_fn = lambda x: best_model.predict_proba(
    pd.DataFrame(x, columns=X.columns)
)[:, 1]
explainer = shap.Explainer(
    predict_fn, background, algorithm="permutation",
    feature_names=list(X.columns)
)
shap_values = explainer(sample, max_evals=2 * X.shape[1] + 1)
shap.plots.beeswarm(shap_values, max_display=12, show=False)
plt.tight_layout()
plt.savefig(RESULTS / "shap_resumo.png", dpi=160, bbox_inches="tight")
plt.show()

## Conclusões e justificativa das decisões
- **Acurácia** não é métrica principal: o baseline trivial já supera 99,8%.
- **AP / curva PR** permitem comparar a qualidade de ordenação das probabilidades.
- **Recall** prioriza identificar fraudes; **precision** limita o volume de alertas.
- **Limiar** é escolhido em validação, não no teste.
- **SHAP** explica as previsões, mas as variáveis PCA não são interpretáveis como atributos originais.
- **Próximos passos:** validação temporal, calibragem de probabilidades, custo de investigação, monitorização de drift e revisão humana.

> Os números finais devem ser preenchidos com os resultados reais após executar todas as células. Não utilize métricas de outros projetos como se fossem deste experimento.